<div style="background-color:#0e2233;background-image:linear-gradient(135deg,#0b1a29,#123049);color:#ffffff;padding:26px 28px 24px;border-radius:12px;border:1px solid #2a4a66;">
<div style="color:#a0d2ff;font-size:0.78em;font-weight:600;letter-spacing:0.18em;">AI &amp; ROBOTICS · NHÁNH AI ENGINEER · NOTEBOOK THỰC HÀNH</div>
<div style="width:56px;height:4px;background-color:#E07A1F;margin:14px 0 12px;"></div>
<h1 style="color:#ffffff;font-size:2.3em;line-height:1.15;margin:0;border:0;">Chương 10 · Hồi quy tuyến tính</h1>
<p style="color:#cfe6ff;font-size:1.12em;line-height:1.5;margin:10px 0 0;">Từ bốn chấm điểm trên giấy đến vòng lặp gradient descent tự viết bằng NumPy. Vòng lặp này là trái tim của mọi mạng nơ-ron bạn sẽ gặp sau này.</p>
</div>

### Sau notebook này bạn sẽ

- Tự viết được gradient descent bằng NumPy, mỗi dòng code đều hiểu nó làm gì.
- Tính tay được một bước gradient descent rồi để máy kiểm chứng từng con số.
- Thấy tận mắt learning rate quá nhỏ, vừa, quá lớn khác nhau thế nào, và loss phát nổ ra sao.
- Đối chiếu được kết quả tự viết với thư viện scikit-learn.
- Đọc được đường hồi quy, đường cong loss và đồ thị phần dư để biết mô hình đang khỏe hay đang bệnh.

<div>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Thời gian chạy:</b> khoảng 2 phút</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Runtime:</b> CPU là đủ, không cần GPU</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Trang web của chương:</b> <a href="https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch10-hoi-quy-tuyen-tinh/01-truc-giac/" style="color:#1F5F8B;">mở bài đọc</a></span>
</div>

### Mục lục

1. [Trang 1 · Bài toán và trực giác](#trang-1)
2. [Trang 2 · Toán từng bước](#trang-2)
3. [Trang 3 · Tính tay một ví dụ nhỏ](#trang-3)
4. [Trang 4 · Code](#trang-4)
5. [Trang 5 · Đọc kết quả](#trang-5)
6. [Trang 6 · Bài tập](#trang-6)

### Cách chạy

Bấm vào một ô code rồi nhấn **Shift + Enter** để chạy ô đó và nhảy sang ô kế tiếp. Muốn chạy hết một lượt thì chọn menu **Runtime > Run all**. Các ô nên chạy theo thứ tự từ trên xuống, vì ô sau dùng biến của ô trước.

Notebook đi cùng các trang web của chương. Mỗi phần dưới đây mang đúng tên trang tương ứng: đọc phần giải thích trên web trước, rồi chạy ô ở đây để tự kiểm chứng.

<a name="trang-1" id="trang-1"></a>

## Trang 1 · Bài toán và trực giác

Mỗi tuần tự học thêm 1 giờ thì điểm thi tăng thêm bao nhiêu? Phần này nạp dữ liệu của 4 bạn sinh viên và vẽ thành 4 chấm để nhìn bằng mắt trước khi tính.

*Đọc kèm trên web: [Trang 1 · Bài toán và trực giác](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch10-hoi-quy-tuyen-tinh/01-truc-giac/)*

**Bảng 10.1.** Dữ liệu của bốn sinh viên, dùng xuyên suốt chương.

| Sinh viên | Giờ tự học mỗi tuần ($x$) | Điểm thi ($y$) |
|:---:|:---:|:---:|
| A | 1 | 2 |
| B | 2 | 4 |
| C | 3 | 5 |
| D | 4 | 7 |

Cả bảng là **dữ liệu huấn luyện** (training data), thứ máy nhìn vào để học. Số giờ học $x$ là một **đặc trưng** (feature). Điểm thi $y$, thứ ta muốn đoán, là **nhãn** (label).

In [ ]:
import numpy as np               # NumPy: thư viện tính toán trên cả mảng số cùng một lúc
import matplotlib.pyplot as plt  # thư viện vẽ biểu đồ

x = np.array([1, 2, 3, 4], dtype=float)  # giờ tự học mỗi tuần của 4 bạn A, B, C, D
y = np.array([2, 4, 5, 7], dtype=float)  # điểm thi tương ứng của 4 bạn
n = len(x)                               # số mẫu dữ liệu, ở đây n = 4

plt.figure(figsize=(5, 3.6))                         # khung hình rộng 5 inch, cao 3,6 inch
plt.scatter(x, y, s=80, color="#E07A1F", zorder=3)   # 4 chấm cam; zorder=3 để chấm nằm trên lưới
for ten, xi, yi in zip("ABCD", x, y):                # ghép tên từng bạn với tọa độ của bạn đó
    plt.annotate(ten, (xi, yi), xytext=(7, -12), textcoords="offset points")  # ghi tên lệch xuống bên phải chấm
plt.xlabel("Giờ tự học mỗi tuần (x)")  # tên trục ngang
plt.ylabel("Điểm thi (y)")             # tên trục dọc
plt.xlim(0, 5)                         # trục ngang từ 0 đến 5 giờ
plt.ylim(0, 8)                         # trục dọc từ 0 đến 8 điểm
plt.grid(alpha=0.3)                    # lưới mờ để dễ dóng số
plt.show()                             # hiển thị hình

<div style="border-left:6px solid #2E8B57;background-color:#EAF6EF;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#1E6B42;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">ĐỌC KẾT QUẢ</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Bốn chấm đi lên từ trái sang phải: ai học nhiều thì điểm cao hơn.</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Chúng <b>gần như</b> nằm trên một đường thẳng, nhưng không thẳng hàng hẳn. Không có đường thẳng nào đi qua đúng cả bốn chấm.</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Vậy ta tìm đường thẳng đi <b>gần nhất có thể</b> với cả bốn chấm. Việc đó gọi là <b>hồi quy tuyến tính</b> (linear regression).</p>
</div>

Đường thẳng đó là một **mô hình** (model): nhận số giờ học, trả ra điểm đoán.

$$\hat y = w x + b$$

- $\hat y$ (đọc là "y mũ") là điểm **đoán**, gọi là dự đoán (prediction).
- $w$ là độ dốc: học thêm 1 giờ thì điểm đoán tăng $w$ điểm. Nó gọi là **trọng số** (weight).
- $b$ là điểm đoán khi không học giờ nào. Nó gọi là **hệ số chặn** (bias, intercept).

$w$ và $b$ là hai núm vặn mà máy tự chỉnh khi học, gọi là **tham số** (parameter). Cả chương chỉ xoay quanh một việc: **tìm $w$ và $b$ tốt nhất**.

<div style="border-left:6px solid #1F5F8B;background-color:#E9F1F8;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#1F5F8B;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">TRỰC GIÁC</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Tưởng tượng bốn chấm là bốn cây đinh đóng trên tường, còn đường thẳng là một thanh gỗ.</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Mỗi cây đinh nối với thanh gỗ bằng một sợi dây thun dựng thẳng đứng. Dây càng dài thì kéo càng mạnh. Thanh gỗ bị kéo qua kéo lại, rồi nằm yên ở chỗ các lực cân bằng.</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Chỗ thanh gỗ nằm yên chính là đường hồi quy. Độ dài mỗi sợi dây là chỗ đường thẳng đoán sai cho bạn đó, gọi là <b>phần dư</b> (residual).</p>
</div>

<a name="trang-2" id="trang-2"></a>

## Trang 2 · Toán từng bước

Gom bốn sợi dây thun thành một con số để biết đường thẳng tốt hay dở, rồi tìm cách xuống đáy của con số đó.

*Đọc kèm trên web: [Trang 2 · Toán từng bước](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch10-hoi-quy-tuyen-tinh/02-toan/)*

Phần dư của bạn thứ $i$ là điểm thật trừ điểm đoán: $e_i = y_i - \hat y_i$. Bình phương từng phần dư rồi lấy trung bình, ta được **sai số bình phương trung bình** (mean squared error, MSE). Đây là **hàm mất mát** (loss function) của chương, gọi tắt là loss:

$$L(w, b) = \frac{1}{n} \sum_{i=1}^{n} e_i^2 \qquad (10.1)$$

Loss càng nhỏ, đường thẳng càng gần dữ liệu. **Học nghĩa là vặn $w$, $b$ để loss nhỏ nhất.**

In [ ]:
def mse(w, b):               # MSE của đường thẳng y_hat = w*x + b, công thức (10.1)
    e = y - (w * x + b)      # phần dư của từng bạn: điểm thật trừ điểm đoán
    return np.mean(e ** 2)   # bình phương từng phần dư rồi lấy trung bình

print(f"Đường y = 0         : MSE = {mse(0.0, 0.0):.4f}")  # chưa học gì: đoán ai cũng 0 điểm
print(f"Đường y = 1,5x + 1  : MSE = {mse(1.5, 1.0):.4f}")  # một đường đoán đại cho gần
print(f"Đường y = 1,6x + 0,5: MSE = {mse(1.6, 0.5):.4f}")  # đường tốt nhất, các trang sau sẽ tìm ra

Output mong đợi:

```text
Đường y = 0         : MSE = 23.5000
Đường y = 1,5x + 1  : MSE = 0.1250
Đường y = 1,6x + 0,5: MSE = 0.0500
```

<div style="border-left:6px solid #2E8B57;background-color:#EAF6EF;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#1E6B42;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">ĐỌC KẾT QUẢ</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Đường nằm ngang y = 0 sai rất xa: MSE 23,5. Đoán đại một đường gần gần đã xuống 0,125.</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Đường y = 1,6x + 0,5 cho MSE = 0,05, thấp nhất có thể với 4 chấm này. Không đường thẳng nào xuống được 0, vì 4 chấm không thẳng hàng.</p>
</div>

<div style="border-left:6px solid #E07A1F;background-color:#FDF1E6;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#9A4F0E;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">VÌ SAO LẠI LÀ BÌNH PHƯƠNG MÀ KHÔNG PHẢI TRỊ TUYỆT ĐỐI?</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>Một, bình phương trơn.</b> Đồ thị của e² là một cái chén cong đều. Đồ thị của |e| gãy ở 0, mà lát nữa ta dò đáy bằng độ dốc, tại chỗ gãy thì không đo được độ dốc.</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>Hai, bình phương phạt nặng sai số lớn.</b> Sai 1 điểm bị tính 1, sai 3 điểm bị tính 9. Mô hình cố không để bạn nào bị đoán lệch quá xa.</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>Cái giá phải trả.</b> MSE dễ bị một <b>điểm ngoại lai</b> (outlier) kéo lệch. Ô code ngay dưới cho bạn thấy: chỉ một con số gõ nhầm là đường thẳng gần như nằm ngang.</p>
</div>

In [ ]:
def do_doc(xx, yy):                                # độ dốc tốt nhất, dòng đầu của công thức (10.4)
    lech_x = xx - xx.mean()                        # mỗi bạn học lệch khỏi trung bình bao nhiêu giờ
    lech_y = yy - yy.mean()                        # mỗi bạn được lệch khỏi trung bình bao nhiêu điểm
    return np.sum(lech_x * lech_y) / np.sum(lech_x ** 2)  # tử số chia mẫu số

x_nham = np.array([1, 2, 3, 40], dtype=float)      # bạn D học 4 giờ nhưng gõ nhầm thành 40
print(f"Gõ đúng : w = {do_doc(x, y):.4f}")         # dữ liệu chuẩn
print(f"Gõ nhầm : w = {do_doc(x_nham, y):.4f}")    # chỉ sai đúng một con số

Output mong đợi:

```text
Gõ đúng : w = 1.6000
Gõ nhầm : w = 0.0903
```

<div style="border-left:6px solid #2E8B57;background-color:#EAF6EF;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#1E6B42;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">ĐỌC KẾT QUẢ</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Chỉ một con số gõ nhầm, độ dốc tụt từ 1,6 xuống còn khoảng 0,09: đường thẳng gần như nằm ngang.</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Bạn D ở x = 40 nằm rất xa ba bạn kia. Phần dư của D rất lớn, bình phương lên còn lớn hơn nữa, nên cả đường thẳng phải xoay theo để chiều D. Gặp loss tăng vọt bất thường, hãy kiểm tra dữ liệu trước.</p>
</div>

### Đi xuống đáy của loss

Lấy **đạo hàm riêng** (partial derivative) của $L$ theo từng tham số, ta biết nhích $w$, $b$ theo hướng nào thì loss tăng:

$$\frac{\partial L}{\partial w} = -\frac{2}{n} \sum_{i} x_i e_i \quad (10.2) \qquad \frac{\partial L}{\partial b} = -\frac{2}{n} \sum_{i} e_i \quad (10.3)$$

Gộp hai đạo hàm riêng thành một cặp số, ta có **gradient**: mũi tên chỉ hướng lên dốc nhanh nhất. Có hai con đường xuống đáy.

**Con đường 1, nghiệm đóng** (closed-form solution): cho hai đạo hàm bằng 0 rồi giải thẳng ra đáp án.

$$w^{*} = \frac{\sum_{i} (x_i - \bar x)(y_i - \bar y)}{\sum_{i} (x_i - \bar x)^2}, \qquad b^{*} = \bar y - w^{*} \bar x \qquad (10.4)$$

**Con đường 2, gradient descent**: đi từng bước ngược hướng gradient.

$$w \leftarrow w - \eta \frac{\partial L}{\partial w}, \qquad b \leftarrow b - \eta \frac{\partial L}{\partial b} \qquad (10.5)$$

Chữ $\eta$ (đọc là "ê-ta") là **learning rate**, độ dài mỗi bước chân. Ta tự chọn nó trước khi chạy.

<div style="border-left:6px solid #E07A1F;background-color:#FDF1E6;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#9A4F0E;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">VÌ SAO LẠI LÀ η = 0,05?</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Learning rate là <b>độ dài mỗi bước chân</b> khi đi xuống dốc.</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Bước quá ngắn: đi đúng đường nhưng lâu tới đáy. Bước quá dài: nhảy vọt qua đáy, sang sườn bên kia ở chỗ còn cao hơn chỗ cũ, càng nhảy càng xa.</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Với dữ liệu 4 bạn sinh viên, bước dài hơn khoảng 0,12 là bắt đầu nhảy loạn. 0,05 nằm an toàn dưới ngưỡng đó mà vẫn đủ nhanh. Trang 5 cho bạn tự thấy cả ba trường hợp.</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Ngưỡng này phụ thuộc vào con số trong dữ liệu. Đổi giờ sang phút (số lớn gấp 60) thì cùng bước chân 0,05 sẽ nổ ngay. Vì vậy người ta luôn <b>chuẩn hóa</b> (feature scaling) dữ liệu về cỡ nhỏ trước khi huấn luyện.</p>
</div>

<a name="trang-3" id="trang-3"></a>

## Trang 3 · Tính tay một ví dụ nhỏ

Thay số vào công thức bằng tay, đi cả hai con đường của Trang 2. Hãy tính ra giấy trước, rồi chạy ô code để máy chấm giúp.

*Đọc kèm trên web: [Trang 3 · Tính tay một ví dụ nhỏ](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch10-hoi-quy-tuyen-tinh/03-tinh-tay/)*

### Con đường 1: nghiệm đóng

In [ ]:
x_tb, y_tb = x.mean(), y.mean()                        # hai trung bình: 2,5 giờ và 4,5 điểm
lech_x = x - x_tb                                      # độ lệch giờ học của từng bạn
lech_y = y - y_tb                                      # độ lệch điểm thi của từng bạn
w_sao = np.sum(lech_x * lech_y) / np.sum(lech_x ** 2)  # công thức (10.4), dòng đầu
b_sao = y_tb - w_sao * x_tb                            # công thức (10.4), dòng hai

print("Độ lệch x:", lech_x)                                            # cộng lại phải bằng 0
print("Độ lệch y:", lech_y)                                            # cộng lại cũng bằng 0
print("Tử số    :", np.sum(lech_x * lech_y))                           # tổng các tích độ lệch
print("Mẫu số   :", np.sum(lech_x ** 2))                               # tổng bình phương độ lệch x
print(f"w* = {w_sao:.4f}, b* = {b_sao:.4f}, MSE = {mse(w_sao, b_sao):.4f}")  # đáp án và loss tại đáy

Output mong đợi:

```text
Độ lệch x: [-1.5 -0.5  0.5  1.5]
Độ lệch y: [-2.5 -0.5  0.5  2.5]
Tử số    : 8.0
Mẫu số   : 5.0
w* = 1.6000, b* = 0.5000, MSE = 0.0500
```

<div style="border-left:6px solid #2E8B57;background-color:#EAF6EF;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#1E6B42;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">ĐỌC KẾT QUẢ</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Hai cột độ lệch đều cộng lại bằng 0. Đó là cách tự kiểm tra nhanh khi tính tay.</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Tử số 8, mẫu số 5, nên w* = 8 / 5 = 1,6 và b* = 4,5 - 1,6 × 2,5 = 0,5. Học thêm 1 giờ, điểm đoán tăng 1,6 điểm.</p>
</div>

### Con đường 2: một bước gradient descent

Bắt đầu từ $w = b = 0$, bước chân $\eta = 0{,}05$. Đoán, đo sai số, tính gradient, cập nhật.

In [ ]:
w, b = 0.0, 0.0                       # chưa biết gì: bắt đầu từ đường nằm ngang y = 0
lr = 0.05                             # learning rate η
e = y - (w * x + b)                   # phần dư: đang đoán 0 nên bằng đúng điểm thật
grad_w = -2 / n * np.sum(x * e)       # công thức (10.2)
grad_b = -2 / n * np.sum(e)           # công thức (10.3)
print("Phần dư     :", e)             # 4 phần dư trước khi cập nhật
print("Gradient    :", grad_w, grad_b)  # hai đạo hàm riêng
w = w - lr * grad_w                   # công thức (10.5) cho w
b = b - lr * grad_b                   # công thức (10.5) cho b
print(f"Sau 1 bước  : w = {w:.4f}, b = {b:.4f}, MSE = {mse(w, b):.6f}")  # loss của đường thẳng mới

Output mong đợi:

```text
Phần dư     : [2. 4. 5. 7.]
Gradient    : -26.5 -9.0
Sau 1 bước  : w = 1.3250, b = 0.4500, MSE = 0.688437
```

<div style="border-left:6px solid #2E8B57;background-color:#EAF6EF;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#1E6B42;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">ĐỌC KẾT QUẢ</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Chỉ một bước, loss từ 23,5 xuống 0,688, tức giảm hơn 97%.</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Bước đầu nhảy xa vì quãng đi mỗi bước là η × độ dốc. Ở xa đáy thì dốc rất đứng (-26,5), nên quãng đi dài. Càng gần đáy, dốc càng thoải, quãng đi tự ngắn lại.</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Sau một bước: w = 1,325 (đích là 1,6), b = 0,45 (đích là 0,5). Gần rồi, nhưng chưa tới.</p>
</div>

<div style="border-left:6px solid #C0392B;background-color:#FBECEA;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#962D22;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">LỖI HAY GẶP</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>Đảo chiều phần dư.</b> Chương này định nghĩa e = y - ŷ (thật trừ đoán). Có sách viết ngược lại. Trộn định nghĩa sách này với công thức sách kia là bước lên dốc thay vì xuống dốc.</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>Quên rằng phần dư đổi sau mỗi bước.</b> Bước 2 phải tính lại phần dư từ w = 1,325 và b = 0,45, không dùng lại <code style="background:#ffffffb3;color:#2b2b2b;padding:1px 5px;border-radius:4px;white-space:nowrap;">[2, 4, 5, 7]</code>.</p>
</div>

<a name="trang-4" id="trang-4"></a>

## Trang 4 · Code

Giao việc lặp cho máy: vẫn đúng các phép tính của Trang 3, chỉ là chạy 200 lần. Sau đó đối chiếu với thư viện scikit-learn.

*Đọc kèm trên web: [Trang 4 · Code](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch10-hoi-quy-tuyen-tinh/04-code/)*

### Gradient descent tự viết bằng NumPy

Hãy đọc chậm, đối chiếu từng dòng với công thức (10.1) đến (10.5).

In [ ]:
import numpy as np  # NumPy: thư viện tính toán trên cả mảng số cùng một lúc

# Dữ liệu Bảng 10.1. dtype=float vì lát nữa sẽ nhân chia ra số thập phân
x = np.array([1, 2, 3, 4], dtype=float)  # giờ tự học mỗi tuần của 4 bạn A, B, C, D
y = np.array([2, 4, 5, 7], dtype=float)  # điểm thi tương ứng của 4 bạn
n = len(x)                               # số mẫu dữ liệu, ở đây n = 4

w, b = 0.0, 0.0  # khởi tạo: chưa biết gì nên bắt đầu từ đường nằm ngang y = 0
lr = 0.05        # learning rate η, xem khung "Vì sao lại là η = 0,05?" ở trang Toán

for buoc in range(201):        # lặp 201 lần (bước 0 đến 200) để in được cả bước 0 lẫn bước 200
    y_hat = w * x + b          # đoán cho cả 4 bạn cùng lúc: NumPy nhân w với từng phần tử của x
    e = y - y_hat              # phần dư e_i = y_i - ŷ_i, cũng là một mảng 4 phần tử
    loss = np.mean(e ** 2)     # MSE, công thức (10.1): bình phương từng phần dư rồi lấy trung bình
    if buoc % 50 == 0:         # % là phép chia lấy dư: đúng ở bước 0, 50, 100, 150, 200
        print(f"Bước {buoc:3d} | w = {w:.4f} | b = {b:.4f} | MSE = {loss:.6f}")  # w, b in 4 chữ số thập phân, MSE in 6
    grad_w = -2 / n * np.sum(x * e)  # công thức (10.2): x * e nhân từng cặp phần tử rồi cộng lại
    grad_b = -2 / n * np.sum(e)      # công thức (10.3): cộng các phần dư
    w = w - lr * grad_w              # công thức (10.5): bước ngược hướng gradient
    b = b - lr * grad_b              # công thức (10.5) cho b, cùng độ dài bước lr

Output mong đợi, khớp với output in trên trang 4 của web:

```text
Bước   0 | w = 0.0000 | b = 0.0000 | MSE = 23.500000
Bước  50 | w = 1.5937 | b = 0.5186 | MSE = 0.050058
Bước 100 | w = 1.5970 | b = 0.5088 | MSE = 0.050013
Bước 150 | w = 1.5986 | b = 0.5041 | MSE = 0.050003
Bước 200 | w = 1.5993 | b = 0.5019 | MSE = 0.050001
```

<div style="border-left:6px solid #2E8B57;background-color:#EAF6EF;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#1E6B42;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">ĐỌC KẾT QUẢ</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Dòng bước 0 khớp tính tay: w = b = 0, MSE = 23,5.</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Dòng bước 200: w ≈ 1,5993, b ≈ 0,5019, MSE ≈ 0,050001. Gần sát đáp án w = 1,6, b = 0,5, MSE = 0,05 của nghiệm đóng.</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Để ý điều lạ: từ bước 50, MSE đã gần như chạm đáy (0,050058), nhưng w và b vẫn còn nhích qua các bước 100, 150, 200. Đáy của loss không phải cái hố tròn mà là một thung lũng dài và rất thoải. Đi dọc thung lũng, loss gần như không đổi, nên gradient nhỏ và bước đi ngắn.</p>
</div>

<div style="border-left:6px solid #7A6A58;background-color:#F4F1EC;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#5C4E3F;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">THỬ SỬA</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>1. Bước chân quá dài.</b> Trong ô trên, đổi <code style="background:#ffffffb3;color:#2b2b2b;padding:1px 5px;border-radius:4px;white-space:nowrap;">lr = 0.05</code> thành <code style="background:#ffffffb3;color:#2b2b2b;padding:1px 5px;border-radius:4px;white-space:nowrap;">lr = 0.135</code> rồi chạy lại. Đoán trước: loss sẽ thế nào?</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>2. Bước chân dài hơn mà vẫn an toàn.</b> Đổi thành <code style="background:#ffffffb3;color:#2b2b2b;padding:1px 5px;border-radius:4px;white-space:nowrap;">lr = 0.1</code>. Ở bước 200, w và b gần đáp án hơn hay xa hơn so với lr = 0,05?</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>3. Kiểm chứng tính tay.</b> Đổi <code style="background:#ffffffb3;color:#2b2b2b;padding:1px 5px;border-radius:4px;white-space:nowrap;">if buoc % 50 == 0:</code> thành <code style="background:#ffffffb3;color:#2b2b2b;padding:1px 5px;border-radius:4px;white-space:nowrap;">if buoc <= 1:</code>. Dòng bước 1 phải khớp đúng con số ở Trang 3.</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Thử xong nhớ trả về <code style="background:#ffffffb3;color:#2b2b2b;padding:1px 5px;border-radius:4px;white-space:nowrap;">lr = 0.05</code> và <code style="background:#ffffffb3;color:#2b2b2b;padding:1px 5px;border-radius:4px;white-space:nowrap;">if buoc % 50 == 0:</code> rồi chạy lại ô, vì các ô sau dùng tiếp biến của ô này.</p>
</div>

<details><summary>Xem điều sẽ xảy ra</summary>

**1.** Với `lr = 0.135`, loss tăng ngay từ bước 1 và **phát nổ** (diverge). Dòng bước 50 là:

```text
Bước  50 | w = -133475.1083 | b = -45397.8523 | MSE = 165979360957.275085
```

MSE hơn 165 tỷ, $w$ thành số âm cực lớn. Mỗi bước nhảy quá xa, vượt qua đáy sang vách bên kia ở chỗ còn cao hơn chỗ cũ.

**2.** Với `lr = 0.1`, bước 200 in ra:

```text
Bước 200 | w = 1.6000 | b = 0.5001 | MSE = 0.050000
```

Gần đáp án hơn lr = 0,05: bước chân dài hơn thì đi dọc thung lũng nhanh hơn, miễn là còn dưới ngưỡng khoảng 0,12.

**3.** Hai dòng đầu là:

```text
Bước   0 | w = 0.0000 | b = 0.0000 | MSE = 23.500000
Bước   1 | w = 1.3250 | b = 0.4500 | MSE = 0.688437
```

Đúng $w = 1{,}325$, $b = 0{,}45$, MSE $\approx 0{,}688$ như Trang 3.

</details>

### Mấy chỗ đáng dừng lại trong code

**`x * e` không phải phép nhân ma trận.** Với mảng NumPy, `*` nhân **từng cặp** phần tử cùng vị trí:

```python
np.array([1, 2, 3, 4]) * np.array([2, 4, 5, 7])  # ra array([ 2,  8, 15, 28])
```

Cộng lại được 53, đúng $\sum x_i e_i$ ở bước đầu.

**Không có vòng lặp qua từng bạn.** Dòng `y_hat = w * x + b` đoán cho cả 4 bạn bằng một phép tính. Viết phép tính cho cả mảng một lần như vậy gọi là **vector hóa** (vectorization), nhanh hơn vòng `for` của Python rất nhiều khi dữ liệu lớn.

**Vòng `for buoc` thì không bỏ được.** Mỗi bước cần $w$, $b$ của bước trước, nên phải xong bước này mới biết bước sau đứng ở đâu.

### Đối chiếu với scikit-learn

Ngoài đời, không ai tự viết vòng lặp cho hồi quy tuyến tính, ta dùng thư viện. Nhưng chính vì đã tự viết, giờ bạn biết thư viện đang làm gì bên trong.

In [ ]:
from sklearn.linear_model import LinearRegression  # mô hình hồi quy tuyến tính có sẵn của scikit-learn

X = x.reshape(-1, 1)          # sklearn cần X là bảng (số mẫu, số đặc trưng): đổi hình dạng từ (4,) thành (4, 1)
mo_hinh = LinearRegression()  # tạo mô hình rỗng, chưa học gì
mo_hinh.fit(X, y)             # "học": giải thẳng bằng nghiệm đóng, không lặp từng bước
print("w =", mo_hinh.coef_[0])      # coef_ là mảng các w, mỗi đặc trưng một số; ở đây chỉ có 1
print("b =", mo_hinh.intercept_)    # intercept_ là hệ số chặn b
print("R² =", mo_hinh.score(X, y))  # score trả về R², trang Đọc kết quả sẽ giải thích

Output mong đợi:

```text
w = 1.5999999999999996
b = 0.5000000000000009
R² = 0.9846153846153847
```

<div style="border-left:6px solid #2E8B57;background-color:#EAF6EF;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#1E6B42;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">ĐỌC KẾT QUẢ</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">w và b trùng nghiệm đóng: 1,6 và 0,5.</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Số <code style="background:#ffffffb3;color:#2b2b2b;padding:1px 5px;border-radius:4px;white-space:nowrap;">1.5999999999999996</code> chính là 1,6. Máy lưu số thực ở dạng <b>dấu phẩy động</b> (floating-point), khoảng 16 chữ số có nghĩa, nên lệch ở chữ số cuối là bình thường. Khi so hai số thực trong code, đừng dùng <code style="background:#ffffffb3;color:#2b2b2b;padding:1px 5px;border-radius:4px;white-space:nowrap;">==</code>, hãy xem chúng có đủ gần nhau không.</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Dòng cuối là <b>R bình phương</b> (R-squared), điểm số của đường hồi quy trên thang 0 đến 1. Trang 5 sẽ tính tay ra đúng con số này.</p>
</div>

<div style="border-left:6px solid #C0392B;background-color:#FBECEA;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#962D22;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">LỖI HAY GẶP</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>Quên dấu trừ khi cập nhật.</b> Viết <code style="background:#ffffffb3;color:#2b2b2b;padding:1px 5px;border-radius:4px;white-space:nowrap;">w = w + lr * grad_w</code> là đi <b>theo</b> hướng gradient, tức leo lên dốc. Chạy thử: đến bước 50, MSE đã vượt 10<sup>27</sup>.</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>Quên chia cho n.</b> Bỏ <code style="background:#ffffffb3;color:#2b2b2b;padding:1px 5px;border-radius:4px;white-space:nowrap;">/ n</code> thì bước chân thực tế dài gấp n lần. Với 4 bạn, bước chân thành 0,2, vượt ngưỡng 0,12, và loss phát nổ ngay.</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>Dùng <code style="background:#ffffffb3;color:#2b2b2b;padding:1px 5px;border-radius:4px;white-space:nowrap;">y_hat</code> của bước trước.</b> Phải tính lại <code style="background:#ffffffb3;color:#2b2b2b;padding:1px 5px;border-radius:4px;white-space:nowrap;">y_hat</code> và <code style="background:#ffffffb3;color:#2b2b2b;padding:1px 5px;border-radius:4px;white-space:nowrap;">e</code> ở đầu mỗi bước, sau khi w, b đã đổi. Đặt sai chỗ là gradient luôn trễ một nhịp.</p>
</div>

<a name="trang-5" id="trang-5"></a>

## Trang 5 · Đọc kết quả

Một cặp số w, b chưa nói được mô hình tốt hay dở. Phần này vẽ và đọc ba biểu đồ: đường hồi quy, đường cong loss và đồ thị phần dư.

*Đọc kèm trên web: [Trang 5 · Đọc kết quả](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch10-hoi-quy-tuyen-tinh/05-doc-ket-qua/)*

### Đường hồi quy và phần dư

In [ ]:
w_kq, b_kq = mo_hinh.coef_[0], mo_hinh.intercept_  # w, b học được ở ô scikit-learn
y_doan = w_kq * x + b_kq                           # điểm đoán cho 4 bạn
e_kq = y - y_doan                                  # phần dư của từng bạn

x_ve = np.linspace(0.5, 4.5, 50)                   # 50 điểm cách đều để vẽ đường thẳng cho mượt
plt.figure(figsize=(5.5, 3.8))                     # khung hình
plt.plot(x_ve, w_kq * x_ve + b_kq, color="#1F5F8B", label="đường hồi quy ŷ = 1,6x + 0,5")  # đường xanh
plt.vlines(x, y_doan, y, colors="#C0392B", linestyles="--", linewidth=2, label="phần dư")  # đoạn đỏ nối chấm với đường
plt.scatter(x, y, s=80, color="#E07A1F", zorder=3, label="điểm thật")          # 4 chấm cam
for xi, yi, ei in zip(x, y, e_kq):                 # ghi độ dài có dấu của từng phần dư
    plt.annotate(f"e = {ei:+.1f}", (xi, yi), xytext=(8, 4), textcoords="offset points", fontsize=9)  # nhãn cạnh chấm
plt.xlabel("Giờ tự học mỗi tuần (x)")  # tên trục ngang
plt.ylabel("Điểm thi (y)")             # tên trục dọc
plt.legend()                           # chú thích màu
plt.grid(alpha=0.3)                    # lưới mờ
plt.show()                             # hiển thị hình

print("Phần dư:", np.round(e_kq, 4))                    # làm tròn 4 chữ số cho dễ đọc
print("Tổng phần dư:", round(float(np.sum(e_kq)), 10))  # cộng lại phải bằng 0

Output mong đợi (dưới hình):

```text
Phần dư: [-0.1  0.3 -0.3  0.1]
Tổng phần dư: 0.0
```

<div style="border-left:6px solid #2E8B57;background-color:#EAF6EF;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#1E6B42;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">ĐỌC KẾT QUẢ</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>Trục.</b> Ngang là giờ học, dọc là điểm thi. Chấm cam là 4 bạn, đường xanh là đường mô hình đoán, đoạn đỏ đứt nét là phần dư, chính là "sợi dây thun" ở Trang 1.</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>Điều cần thấy.</b> Phần dư có cả dương lẫn âm và cộng lại đúng bằng 0: -0,1 + 0,3 - 0,3 + 0,1 = 0. Đường thẳng nằm giữa các chấm, không thiên lên hay xuống.</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>Nếu mô hình bệnh.</b> Mọi đoạn đỏ cùng chỉ lên là b chưa học tới nơi. Bên trái chỉ lên, bên phải chỉ xuống là độ dốc sai. Cả hai thường do vòng lặp dừng quá sớm hoặc code có lỗi.</p>
</div>

### Chấm điểm đường thẳng bằng R bình phương

Hình dung một "mô hình lười" (baseline): không nhìn giờ học, đoán ai cũng được điểm trung bình 4,5. R² so mô hình của ta với mô hình lười:

$$R^2 = 1 - \frac{\sum e_i^2}{\sum (y_i - \bar y)^2}$$

In [ ]:
sai_cua_ta = np.sum(e_kq ** 2)           # tổng bình phương phần dư của đường hồi quy
sai_lam_bieng = np.sum((y - y.mean()) ** 2)  # tổng bình phương sai số của mô hình lười đoán 4,5
r2 = 1 - sai_cua_ta / sai_lam_bieng      # phần sai số đã gỡ được
print(f"Mô hình của ta sai : {sai_cua_ta:.4f}")    # khoảng 0,2
print(f"Mô hình lười sai   : {sai_lam_bieng:.4f}")  # đúng 13
print(f"R² = {r2:.6f}")                            # so với số scikit-learn in ở Trang 4

Output mong đợi:

```text
Mô hình của ta sai : 0.2000
Mô hình lười sai   : 13.0000
R² = 0.984615
```

<div style="border-left:6px solid #2E8B57;background-color:#EAF6EF;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#1E6B42;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">ĐỌC KẾT QUẢ</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Mô hình lười sai 13, mô hình của ta chỉ còn sai 0,2. R² = 1 - 0,2 / 13 ≈ 0,985: đường thẳng giải thích được khoảng 98,5% sự chênh lệch điểm giữa bốn bạn. Khớp với số scikit-learn in ra.</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Cẩn thận với câu "học thêm 1 giờ <b>làm</b> điểm tăng 1,6". Hồi quy chỉ đo <b>tương quan</b> (correlation), không chứng minh nhân quả (causation). Và 4 điểm dữ liệu đủ để học cách tính, chưa đủ để kết luận về cả lớp.</p>
</div>

### Đường cong loss và learning rate

Ô dưới chạy gradient descent 40 bước với ba learning rate khác nhau và vẽ MSE theo từng bước. Hình này gọi là **đường cong loss** (loss curve), vẽ trên **thang log** (logarithmic scale): mỗi vạch lớn gấp 10 lần vạch ngay dưới.

In [ ]:
import matplotlib.pyplot as plt  # thư viện vẽ biểu đồ

def chay_gd(lr, so_buoc=40):                # chạy gradient descent từ w = b = 0 với learning rate lr
    """Trả về danh sách MSE ở từng bước."""
    w = b = 0.0                             # cùng điểm xuất phát như ô ở trên
    lich_su = []                            # lưu MSE mỗi bước để vẽ
    for _ in range(so_buoc):                # dấu _ nghĩa là không cần dùng số thứ tự bước
        e = y - (w * x + b)                 # phần dư của đường thẳng hiện tại
        lich_su.append(np.mean(e ** 2))     # ghi lại MSE trước khi cập nhật
        w -= lr * (-2 / n * np.sum(x * e))  # w -= ... là viết gọn của w = w - ...
        b -= lr * (-2 / n * np.sum(e))      # cập nhật b tương tự
    return lich_su                          # trả về danh sách MSE

cac_lr = [0.005, 0.05, 0.135]  # quá nhỏ, vừa, quá lớn: thử sửa các số này
for lr_thu in cac_lr:                                              # chạy lần lượt từng learning rate
    plt.plot(chay_gd(lr_thu), label=f"learning rate = {lr_thu}")  # vẽ đường MSE theo bước lặp
plt.yscale("log")              # thang log vì MSE trải từ 0,05 đến hơn một tỷ
plt.xlabel("Bước lặp")         # tên trục ngang
plt.ylabel("MSE (thang log)")  # tên trục dọc
plt.legend()                   # hiện chú thích màu của từng đường
plt.show()                     # hiển thị hình

<div style="border-left:6px solid #2E8B57;background-color:#EAF6EF;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#1E6B42;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">ĐỌC KẾT QUẢ</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>η = 0,05.</b> Rơi gần như thẳng đứng: từ 23,5 xuống 0,69 ngay bước 1, đến bước 3 còn 0,051, rồi đi ngang. Đi ngang không phải ngừng học: nó đã chạm 0,05, mức thấp nhất một đường thẳng đạt được.</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>η = 0,005.</b> Đi đúng hướng nhưng chậm. Tới bước 39, loss vẫn khoảng 0,076, cao hơn đáy chừng 50%. Learning rate nhỏ quá không sai, chỉ tốn thời gian, mà với mô hình lớn thì thời gian là tiền thuê GPU.</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>η = 0,135.</b> Loss tăng ngay từ bước 1: 23,5 rồi 37, 58, 91, tới bước 39 đã vượt một tỷ. Đường đi lên là một đường thẳng trên thang log, nghĩa là mỗi bước loss bị nhân với cùng một số, khoảng 1,57.</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>Nếu mô hình bệnh.</b> Loss thành <b>nan</b> hoặc <b>inf</b>, hay đi lên như đường η = 0,135: nghi phạm số một là learning rate quá lớn hoặc dữ liệu chưa chuẩn hóa.</p>
</div>

<div style="border-left:6px solid #7A6A58;background-color:#F4F1EC;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#5C4E3F;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">THỬ SỬA</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Đổi <code style="background:#ffffffb3;color:#2b2b2b;padding:1px 5px;border-radius:4px;white-space:nowrap;">cac_lr = [0.005, 0.05, 0.135]</code> thành <code style="background:#ffffffb3;color:#2b2b2b;padding:1px 5px;border-radius:4px;white-space:nowrap;">cac_lr = [0.05, 0.1, 0.12]</code> rồi chạy lại ô trên. Đoán trước: đường nào xuống, đường nào lên?</p>
</div>

<details><summary>Xem điều sẽ xảy ra</summary>

Đường 0,1 vẫn xuống tới đáy, nhưng chậm hơn 0,05 ở mấy bước đầu (bước 1 còn 10,6 thay vì 0,69): nó nảy qua lại hai vách thung lũng. Đường 0,12 đi lên rất chậm, từ 23,5 tới khoảng 32 ở bước 39. 0,12 nằm ngay trên ngưỡng phát nổ của bộ dữ liệu này, đúng như khung "Vì sao lại là η = 0,05?" ở Trang 2.

</details>

### Đồ thị phần dư

Dữ liệu thật có hàng chục đặc trưng, không vẽ nổi đường thẳng trong không gian nhiều chiều. Công cụ chẩn đoán dùng được mọi lúc là **đồ thị phần dư** (residual plot): trục ngang là điểm đoán $\hat y$, trục dọc là phần dư. Ô dưới tạo hai bộ dữ liệu giả lập, mỗi bộ 120 điểm: (a) sinh từ đường thẳng, (b) sinh từ đường cong. Cả hai đều được khớp bằng một đường thẳng.

In [ ]:
rng = np.random.default_rng(42)                     # bộ sinh số ngẫu nhiên, seed 42 để ai chạy cũng ra cùng kết quả
xa = rng.uniform(0, 10, 120)                        # 120 giá trị x rải đều từ 0 đến 10
ya_thang = 1.5 * xa + 2 + rng.normal(0, 1.5, 120)   # (a) đường thẳng cộng nhiễu
ya_cong = 0.25 * xa ** 2 + rng.normal(0, 1.5, 120)  # (b) đường cong cộng nhiễu

fig, truc = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)  # hai hình cạnh nhau, chung trục dọc
for o, yy, ten in zip(truc, (ya_thang, ya_cong), ("(a) dữ liệu thẳng", "(b) dữ liệu cong")):  # lần lượt từng bộ
    mh = LinearRegression().fit(xa.reshape(-1, 1), yy)  # khớp một đường thẳng
    du_doan = mh.predict(xa.reshape(-1, 1))             # điểm đoán ŷ
    o.scatter(du_doan, yy - du_doan, s=14, alpha=0.7, color="#1F5F8B")  # mỗi chấm là một điểm dữ liệu
    o.axhline(0, color="#C0392B")                       # vạch 0: trên là đoán thấp, dưới là đoán cao
    r2_bo = mh.score(xa.reshape(-1, 1), yy)             # R² của bộ này
    o.set_title(f"{ten}, R² = {r2_bo:.3f}")             # tên hình kèm R²
    o.set_xlabel("Điểm đoán ŷ")                         # tên trục ngang
    print(f"{ten}: R² = {r2_bo:.3f}")                   # in R² ra để so với hình
truc[0].set_ylabel("Phần dư y - ŷ")  # tên trục dọc, chỉ cần ghi ở hình trái
plt.show()                            # hiển thị hình

Output mong đợi (trên hình):

```text
(a) dữ liệu thẳng: R² = 0.879
(b) dữ liệu cong: R² = 0.905
```

<div style="border-left:6px solid #2E8B57;background-color:#EAF6EF;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#1E6B42;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">ĐỌC KẾT QUẢ</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>Hình (a), mô hình khỏe.</b> Các chấm rải đều thành một dải ngang quanh vạch 0, không có hình thù gì. Phần dư chỉ còn là nhiễu ngẫu nhiên.</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>Hình (b), mô hình bệnh.</b> Các chấm tạo thành hình chữ U: dương ở hai đầu, âm ở giữa. Đường thẳng đoán sai một cách <b>có hệ thống</b>, dấu hiệu quan hệ thật là đường cong.</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>Bài học bất ngờ.</b> R² của (b) là 0,905, còn cao hơn (a). R² chỉ cho biết đường thẳng bám dữ liệu sát tới đâu, không cho biết dạng mô hình có đúng không. Luôn vẽ phần dư.</p>
</div>

<a name="trang-6" id="trang-6"></a>

## Trang 6 · Bài tập

Ba bài tập, mỗi bài kiểm tra một ý của chương. Làm ra giấy trước, đoán kết quả trước khi tính, rồi mới chạy ô kiểm chứng và mở đáp án.

*Đọc kèm trên web: [Trang 6 · Bài tập](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch10-hoi-quy-tuyen-tinh/06-bai-tap/)*

### Bài 10.1 · Thêm một bạn vào lớp

Bạn E học 5 giờ mỗi tuần và được 6 điểm $(x = 5,\ y = 6)$. Dùng 5 bạn A đến E, tính tay lại $w^{*}$ và $b^{*}$ bằng công thức (10.4).

Trước khi tính, hãy đoán: độ dốc sẽ tăng hay giảm so với 1,6? Sau khi tính, giải thích vì sao kết quả hợp lý khi nhìn vị trí của E so với bốn bạn kia.

In [ ]:
# Ô kiểm chứng Bài 10.1: chỉ chạy SAU khi đã tự tính ra giấy
x5 = np.array([1, 2, 3, 4, 5], dtype=float)            # thêm bạn E học 5 giờ
y5 = np.array([2, 4, 5, 7, 6], dtype=float)            # bạn E được 6 điểm
w5 = do_doc(x5, y5)                                    # công thức (10.4), dòng đầu (hàm viết ở Trang 2)
b5 = y5.mean() - w5 * x5.mean()                        # công thức (10.4), dòng hai
print(f"Trung bình: x = {x5.mean()}, y = {y5.mean()}")  # hai trung bình mới
print(f"w* = {w5:.4f}, b* = {b5:.4f}")                 # đáp án làm tròn 4 chữ số

<details><summary>Xem đáp án</summary>

Trung bình mới: $\bar x = 15/5 = 3$ và $\bar y = 24/5 = 4{,}8$. Mỗi dòng là một bạn, A đến E. Dòng cuối là tổng.

| xᵢ − x̄ | yᵢ − ȳ | tích | (xᵢ − x̄)² |
|:---:|:---:|:---:|:---:|
| −2 | −2,8 | 5,6 | 4 |
| −1 | −0,8 | 0,8 | 1 |
| 0 | 0,2 | 0 | 0 |
| 1 | 2,2 | 2,2 | 1 |
| 2 | 1,2 | 2,4 | 4 |
| **0** | **0** | **11** | **10** |

$$w^{*} = \frac{11}{10} = 1{,}1, \qquad b^{*} = 4{,}8 - 1{,}1 \times 3 = 1{,}5$$

Ô kiểm chứng in ra:

```text
Trung bình: x = 3.0, y = 4.8
w* = 1.1000, b* = 1.5000
```

Độ dốc **giảm** từ 1,6 xuống 1,1. E học nhiều nhất lớp nhưng điểm lại thấp hơn D. Đường cũ đoán E được $1{,}6 \times 5 + 0{,}5 = 8{,}5$ điểm, mà E chỉ được 6. E lại đứng ở đầu phải, xa trung bình nhất, nên kéo đường thẳng xoay mạnh nhất: đầu phải bị kéo xuống, đường thẳng nằm thoải ra.

</details>

### Bài 10.2 · Bước gradient descent thứ hai

Sau bước 1 ở Trang 3, ta có $w = 1{,}325$ và $b = 0{,}45$. Tính tay bước thứ hai với $\eta = 0{,}05$: phần dư, hai đạo hàm riêng, $w$ và $b$ mới, rồi MSE mới.

In [ ]:
# Ô kiểm chứng Bài 10.2: in bước 0, 1, 2 của gradient descent
w, b = 0.0, 0.0                       # cùng điểm xuất phát như Trang 3
lr = 0.05                             # cùng learning rate
for buoc in range(3):                 # chỉ cần bước 0, 1, 2
    e = y - (w * x + b)               # phần dư của đường thẳng hiện tại
    print(f"Bước {buoc:3d} | w = {w:.4f} | b = {b:.4f} | MSE = {np.mean(e ** 2):.6f}")  # cùng định dạng Trang 4
    w = w - lr * (-2 / n * np.sum(x * e))  # công thức (10.5) với (10.2)
    b = b - lr * (-2 / n * np.sum(e))      # công thức (10.5) với (10.3)

<details><summary>Xem đáp án</summary>

**Phần dư.** Đường thẳng hiện tại đoán 1,775; 3,1; 4,425; 5,75. Trừ khỏi điểm thật 2; 4; 5; 7, ta được 0,225; 0,9; 0,575; 1,25.

**Gradient.** $\sum x_i e_i = 0{,}225 + 1{,}8 + 1{,}725 + 5 = 8{,}75$ và $\sum e_i = 2{,}95$, nên

$$\frac{\partial L}{\partial w} = -\frac{2}{4} \times 8{,}75 = -4{,}375, \qquad \frac{\partial L}{\partial b} = -\frac{2}{4} \times 2{,}95 = -1{,}475$$

**Cập nhật.**

$$w = 1{,}325 + 0{,}05 \times 4{,}375 = 1{,}54375, \qquad b = 0{,}45 + 0{,}05 \times 1{,}475 = 0{,}52375$$

**MSE mới.** Đường thẳng mới đoán 2,0675; 3,61125; 5,155; 6,69875. Phần dư là −0,0675; 0,38875; −0,155; 0,30125. Bình phương rồi lấy trung bình, được MSE ≈ 0,0676.

Ô kiểm chứng in ra:

```text
Bước   0 | w = 0.0000 | b = 0.0000 | MSE = 23.500000
Bước   1 | w = 1.3250 | b = 0.4500 | MSE = 0.688437
Bước   2 | w = 1.5437 | b = 0.5237 | MSE = 0.067615
```

Loss từ 0,688 xuống 0,0676, giảm thêm khoảng 90%. Gradient bước này nhỏ hơn bước đầu khoảng 6 lần (4,375 so với 26,5). Lần đầu xuất hiện phần dư âm ở A và C: đường thẳng đã chen vào giữa các chấm.

Vì sao 0,52375 in ra `0.5237` chứ không phải `0.5238`? Máy lưu số thực ở dạng nhị phân, nên $b$ thực ra là 0,52374999…, và Python làm tròn xuống. Với $w = 1{,}54375$ cũng vậy.

</details>

### Bài 10.3 · Đổi giờ sang phút

Giữ nguyên $\eta = 0{,}05$ nhưng tính giờ học bằng phút, tức nhân $x$ với 60:

```python
x_phut = x * 60  # 60, 120, 180, 240 phút
```

Trước khi chạy ô dưới, hãy đoán và viết ra: output sẽ trông thế nào?

In [ ]:
# Ô kiểm chứng Bài 10.3: ô này CỐ Ý làm loss phát nổ, NumPy sẽ in thêm cảnh báo overflow
x_phut = x * 60                       # giờ học đổi sang phút: 60, 120, 180, 240
w, b = 0.0, 0.0                       # cùng điểm xuất phát
lr = 0.05                             # cùng learning rate như Trang 4
for buoc in range(201):               # cùng 201 bước như Trang 4
    e = y - (w * x_phut + b)          # phần dư với x tính bằng phút
    if buoc % 50 == 0:                # in ở bước 0, 50, 100, 150, 200
        print(f"Bước {buoc:3d} | w = {w:.4f} | b = {b:.4f} | MSE = {np.mean(e ** 2):.6f}")  # cùng định dạng
    w = w - lr * (-2 / n * np.sum(x_phut * e))  # công thức (10.5) với (10.2)
    b = b - lr * (-2 / n * np.sum(e))           # công thức (10.5) với (10.3)

<details><summary>Xem đáp án</summary>

**Output thật.** Dòng bước 0 vẫn là `MSE = 23.500000`, vì $w = b = 0$ thì đơn vị của $x$ chưa ảnh hưởng gì. Dòng bước 50 in ra `MSE = inf`, với $w$ và $b$ là hai số âm dài hơn 160 chữ số. Từ bước 100 trở đi, cả $w$, $b$ lẫn MSE đều là `nan`. NumPy còn in thêm cảnh báo:

```text
RuntimeWarning: overflow encountered
```

**Chuyện gì xảy ra ở giữa.** Loss nổ ngay từ bước 1: khoảng $1{,}7 \times 10^{8}$, rồi $1{,}2 \times 10^{15}$ ở bước 2. Mỗi bước loss nhân khoảng 7,3 triệu lần. Số thực lớn nhất máy lưu được là khoảng $1{,}8 \times 10^{308}$. Tới bước 45, MSE vượt mức đó và máy ghi là `inf` (vô cực). Vô cực trừ vô cực không có kết quả, máy ghi là `nan` (not a number).

**Vì sao.** Số giờ nhân 60, vách của cái bát loss dốc đứng hơn rất nhiều. Ngưỡng an toàn của $\eta$ tụt từ khoảng 0,12 xuống khoảng $3{,}7 \times 10^{-5}$. Bước chân 0,05 dài gấp hơn 1000 lần ngưỡng.

**Sửa thế nào.** Hoặc chọn $\eta$ nhỏ hơn $3{,}7 \times 10^{-5}$, nhưng khi đó $b$ học cực chậm. Hoặc tốt hơn: **chuẩn hóa** $x$ về cỡ nhỏ trước khi huấn luyện, rồi dùng $\eta$ bình thường. Đây là việc ai cũng làm trước khi huấn luyện mạng nơ-ron.

</details>

<div style="border-left:6px solid #6B4C9A;background-color:#F2EEF8;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#553A7E;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">CÂU HỎI PHỎNG VẤN</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Phỏng vấn AI Engineer thường bằng tiếng Anh. Hãy tập trả lời bằng cả hai thứ tiếng; gợi ý cho từng câu nằm ở trang 6 trên web.</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>1.</b> Vì sao MSE nhạy với outlier? <i>Why is MSE sensitive to outliers?</i></p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>2.</b> Khi nào dùng nghiệm đóng, khi nào dùng gradient descent? <i>When would you use the closed-form solution, and when gradient descent?</i></p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>3.</b> Loss đang giảm thì đột ngột thành nan. Ba thứ đầu tiên bạn kiểm tra là gì? <i>Your loss suddenly became NaN. What are the first three things you check?</i></p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>4.</b> Mô hình có R² = 0,95. Bạn có kết luận mô hình đúng không? <i>Your model has R² = 0.95. Can you conclude that the model is correct?</i></p>
</div>

---

**Xong Chương 10.** Bạn vừa đi trọn vòng lặp sẽ gặp lại ở mọi chương sau: đoán, đo sai số, tính gradient, cập nhật. Quay lại [trang web của chương](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch10-hoi-quy-tuyen-tinh/01-truc-giac/) để đọc kỹ phần toán và các khung Đào sâu.